In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
FKLIM Climate Indices Calculation Pipeline (Production-Ready)
Compatible with FKLIM QC Pipeline (02.Obs_iqr) and FastAPI/TimescaleDB system
- Input: QC data with columns: TEMPERATURE_AVG_C, TEMP_24H_TN_C, TEMP_24H_TX_C, RAINFALL_24H_MM
- Output: ETCCDI indices with QC flags, robust trends (no KNN imputation), and national aggregation
"""

import os
import sys
import glob
import logging
from pathlib import Path
#import climate_extremes as ettcdi
import pandas as pd
import numpy as np
import os.path as pa
from scipy.stats import linregress  # ✅ Import benar untuk analisis tren
#import climate_extremes as ettcdi
sys.path.append(os.path.abspath('00.src'))
import ettcdi
from stats import trend_analysis

# Note: query_climate_data tidak digunakan dalam pipeline ini

# ==============================================================================
# DEFINISI EKSPLISIT INDEKS UNTUK ANALISIS TREN (HINDARI KOLOM METADATA/THRESHOLD)
# ==============================================================================

TREND_INDICES = [
    # Temperature indices (ETCCDI)
    'TMm', 'TMx', 'TMn', 'TXm', 'TXx', 'TXn', 'TNx', 'TNn', 'TNm', 'DTR', 'ETR',
    'Tm10P', 'Tm90P', 'Tn10P', 'Tn90P', 'Tx10P', 'Tx90P', 
    'Tm10', 'Tm90', 'Tn10', 'Tn90', 'Tx10', 'Tx90', 
    'WSDI', 'CSDI',
    # Rainfall indices (ETCCDI)
    'PRECTOT', 'HH', 'HH20MM', 'HH50MM', 'HH100MM', 'HH150MM', 
    'FH20', 'FH50', 'FH100', 'FH150', 'R50', 
    'CDD', 'CWD', 'SDII',
    'RX1DAY', 'RX5DAY', 'RX7DAY', 'RX10DAY', 
    'R95P', 'R99P', 'R95Ptot', 'R99Ptot'
]

# ==============================================================================
# HELPER FUNCTIONS (SAFE STRING HANDLING & ROBUST TREND ANALYSIS)
# ==============================================================================

def sanitize_filename(text):
    """Bersihkan string untuk penggunaan sebagai nama file/direktori."""
    if pd.isna(text) or text is None:
        return "UNKNOWN"
    # Ganti karakter ilegal dengan underscore
    for ch in ['\\', '/', ':', '*', '?', '"', '<', '>', '|']:
        text = str(text).replace(ch, '_')
    # Ganti spasi berlebih dengan underscore tunggal
    text = '_'.join(text.split())
    # Batasi panjang maksimal 100 karakter
    return text[:100].strip('_')

In [15]:
# ==============================================================================
# SETUP PROJECT STRUCTURE (KONSISTEN DENGAN FKLIM STANDARD)
# ==============================================================================
workDir = os.getcwd()
dataDir = os.path.join('..', '01.data')
outDir = os.path.join('..', '02.output')
srcDir = os.path.join('00.src')

# Directories sesuai FKLIM standard
obsDir = os.path.join(dataDir,    "01.ObsHomo")          # Data observasi mentah
qcDir = os.path.join(dataDir,     "02.Obs_HybridIQR") # Data hasil QC (IQR/Hybrid)
reportDir = os.path.join(dataDir, "03.Report")    # Report per stasiun
indiceDir = os.path.join(outDir,  "01.indices")    # Indeks klimatologi agregat

In [16]:
# Buckets untuk agregasi nasional
DataBucket  = []
SlopeBucket = []
IndekBucket = []  # ✅ Nama konsisten: "Indek" (bukan "Indices")
# ==============================================================================
# MAIN PROCESSING LOOP
# ==============================================================================
qcDirFile = sorted(glob.glob(pa.join(qcDir, 'FKLIM_QC_DAILY_*.csv')))
if not qcDirFile:
    raise FileNotFoundError(f"Tidak ditemukan file QC di direktori: {qcDir}")
print(f"Memproses {len(qcDirFile)} stasiun dari direktori QC...")
for i, filepath in enumerate(qcDirFile, 1):
    filename = pa.basename(filepath)
    wmo_id = filename.replace('.csv', '').split('_')[-1]
    start_date_meta = filename.replace('.csv', '').split('_')[5]
    end_date_meta = filename.replace('.csv', '').split('_')[6]
    print(f"\n[{i}/{len(qcDirFile)}] Memproses stasiun {wmo_id} ({start_date_meta} - {end_date_meta})...")
    try:
        # Baca data QC
        df = pd.read_csv(filepath, parse_dates=['time'])
        if df.empty:
            print(f"  ⚠️  File kosong untuk stasiun {wmo_id}, dilewati.")
            continue
        # Validasi kolom wajib
        required_cols = ['TEMPERATURE_AVG_C', 'TEMP_24H_TN_C', 'TEMP_24H_TX_C', 'RAINFALL_24H_MM', 
                         'name', 'wmo_id', 'latitude', 'longitude', 'elevasi', 'provinsi', 'kabupaten']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"  ⚠️  Kolom tidak lengkap untuk stasiun {wmo_id}: {missing_cols}, dilewati.")
            continue
        # Hitung indeks temperatur (selalu berhasil kecuali error kritis)
        try:
            temp_indices = ettcdi.idxTemp(
                df, 
                tave="TEMPERATURE_AVG_C", 
                tmin="TEMP_24H_TN_C", 
                tmax="TEMP_24H_TX_C",
                ref_start=1991,
                ref_end=2020
            ).reset_index()
        except Exception as e:
            print(f"  ⚠️  Gagal menghitung indeks temperatur untuk {wmo_id}: {str(e)[:100]}")
            # Buat DataFrame kosong dengan struktur minimal
            years = sorted(df['time'].dt.year.unique())
            temp_indices = pd.DataFrame({'YEAR': years})
        
        # Hitung indeks curah hujan dengan fallback baseline robust
        try:
            ch_indices = ettcdi.idxRain(
                df=df, 
                ch='RAINFALL_24H_MM',
                ref_start=1991,
                ref_end=2020,
                min_wet_days=30
            ).reset_index()
            # Ekstrak QC flag untuk logging
            qc_flag = ch_indices['qc_flag'].iloc[0] if 'qc_flag' in ch_indices.columns else 'UNKNOWN'
            baseline_used = ch_indices['baseline_period'].iloc[0] if 'baseline_period' in ch_indices.columns else 'NONE'
            print(f"  ℹ️  Baseline curah hujan: {baseline_used} | QC: {qc_flag}")
        except Exception as e:
            print(f"  ⚠️  Gagal menghitung indeks curah hujan untuk {wmo_id}: {str(e)[:100]}")
            # Buat DataFrame kosong dengan struktur yang benar
            years = sorted(df['time'].dt.year.unique())
            ch_indices = pd.DataFrame({'YEAR': years})
            for col in ['PRECTOT', 'HH', 'HH20MM', 'HH50MM', 'HH100MM', 'HH150MM', 'FH20', 'FH50', 
                       'FH100', 'FH150', 'R50', 'CDD', 'CWD', 'SDII', 'RX1DAY', 'RX5DAY', 'RX7DAY', 
                       'RX10DAY', 'R95P', 'R99P', 'R95Ptot', 'R99Ptot', 'R95p_threshold_mm', 
                       'R99p_threshold_mm', 'baseline_period', 'qc_flag']:
                ch_indices[col] = np.nan
            ch_indices['qc_flag'] = f'ERROR:{str(e)[:50]}'
            ch_indices['baseline_period'] = 'NONE'
        
        # Gabungkan indeks temperatur dan curah hujan (gunakan how='outer' untuk pertahankan semua tahun)
        if 'YEAR' in temp_indices.columns and 'YEAR' in ch_indices.columns:
            indices = pd.merge(temp_indices, ch_indices, on="YEAR", how='outer').set_index('YEAR')
        else:
            print(f"  ⚠️  Tidak dapat menggabungkan indeks untuk {wmo_id}, struktur YEAR tidak valid.")
            continue
        # === PERBAIKAN KRITIS: Normalisasi index menjadi integer tahun ===
        try:
            # Konversi index ke integer tahun (handle string/object/datetime)
            if isinstance(indices.index, pd.DatetimeIndex):
                indices.index = indices.index.year
            elif pd.api.types.is_string_dtype(indices.index) or pd.api.types.is_object_dtype(indices.index):
                indices.index = indices.index.astype(str).str[:4].astype(int)
            else:
                indices.index = indices.index.astype(int)
        except Exception as e:
            print(f"  ⚠️  Gagal normalisasi index tahun: {e}. Menggunakan index asli.")
        # === FILTER HANYA KOLOM INDEKS YANG VALID UNTUK TREN (HINDARI METADATA) ===
        trend_cols = [col for col in indices.columns if col in TREND_INDICES]
        if not trend_cols:
            print(f"  ⚠️  Tidak ada kolom indeks valid untuk analisis tren (daftar TREND_INDICES tidak cocok dengan output indeks)")
            trends = pd.DataFrame()
        else:
            # Filter hanya kolom yang akan dihitung trennya
            indices_for_trend = indices[trend_cols].copy()
            # Hitung tren hanya untuk kolom numerik dengan variasi data
            try:
                # Filter kolom dengan variasi data yang memadai (>1e-10 std)
                variable_cols = [
                    col for col in trend_cols 
                    if indices_for_trend[col].notna().sum() >= 5 and indices_for_trend[col].std() > 1e-10
                ]
                
                if variable_cols:
                    trends_df = trend_analysis(indices_for_trend[variable_cols])
                    trends = trends_df.T  # Transpose untuk format yang diinginkan
                    
                    # Tambahkan metadata stasiun
                    station_meta = {
                        'NAME': df['name'].iloc[0],
                        'WMO_ID': df['wmo_id'].iloc[0],
                        'START_DATA': df['time'].iloc[0],
                        'END_DATA': df['time'].iloc[-1],
                        'LAT': df['latitude'].iloc[0],
                        'LON': df['longitude'].iloc[0],
                        'ELE': df['elevasi'].iloc[0],
                        'PROV': df['provinsi'].iloc[0],
                        'REGENCY': df['kabupaten'].iloc[0]
                    }
                    for key, value in station_meta.items():
                        trends[key] = value
                    
                    # Urutkan kolom
                    meta_cols = ['NAME', 'WMO_ID', 'START_DATA', 'END_DATA', 'LAT', 'LON', 'ELE', 'PROV', 'REGENCY']
                    data_cols = [col for col in trends.columns if col not in meta_cols]
                    trends = trends[meta_cols + sorted(data_cols)]
                    
                    # === PERBAIKAN KRITIS: SAFE FORMATTING UNTUK DEBUGGING ===
                    valid_slopes = trends.loc['slope'].dropna()
                    if len(valid_slopes) > 0:
                        sample_col = valid_slopes.index[0]
                        sample_val = valid_slopes[sample_col]
                        p_val = trends.loc['p_value', sample_col] if 'p_value' in trends.index else np.nan
                        
                        # Format aman tanpa asumsi tipe data
                        try:
                            sample_str = f"{float(sample_val):.4f}" if pd.notna(sample_val) else "NaN"
                            p_str = f"{float(p_val):.3f}" if pd.notna(p_val) else "NaN"
                            print(f"  ✓ Tren berhasil dihitung: {len(valid_slopes)} variabel dengan slope valid")
                            print(f"    Contoh: {sample_col} = {sample_str} unit/tahun (p={p_str})")
                        except (ValueError, TypeError):
                            print(f"  ✓ Tren berhasil dihitung: {len(valid_slopes)} variabel")
                            print(f"    Contoh nilai: {sample_col} = {sample_val} (non-numerik)")
                    else:
                        print(f"  ⚠️  Tren dihitung tetapi semua slope = NaN (periksa variasi data)")
                else:
                    print(f"  ⚠️  Tidak ada kolom dengan variasi data yang memadai untuk analisis tren")
                    trends = pd.DataFrame()
            except Exception as e:
                print(f"  ⚠️  Gagal menghitung tren: {str(e)[:100]}")
                import traceback
                traceback.print_exc()
                trends = pd.DataFrame()
        
        # Siapkan DataFrame indeks dengan metadata
        indices_reset = indices.reset_index()
        
        # Tambahkan metadata stasiun ke indeks
        station_meta = {
            'NAME': df['name'].iloc[0],
            'WMO_ID': df['wmo_id'].iloc[0],
            'START_DATA': df['time'].iloc[0],
            'END_DATA': df['time'].iloc[-1],
            'LAT': df['latitude'].iloc[0],
            'LON': df['longitude'].iloc[0],
            'ELE': df['elevasi'].iloc[0],
            'PROV': df['provinsi'].iloc[0],
            'REGENCY': df['kabupaten'].iloc[0]
        }
        for key, value in station_meta.items():
            indices_reset[key] = value
        # Urutkan kolom indeks
        index_meta_cols = ['NAME', 'WMO_ID', 'START_DATA', 'END_DATA', 'YEAR', 'LAT', 'LON', 'ELE', 'PROV', 'REGENCY']
        index_data_cols = [col for col in indices_reset.columns if col not in index_meta_cols]
        indices_reset = indices_reset[index_meta_cols + sorted(index_data_cols)]
        # === PENANGANAN NAMA FILE AMAN ===
        station_name_clean = sanitize_filename(df['name'].iloc[0])
        station_dir = Path(reportDir) / f"{df['wmo_id'].iloc[0]}_{station_name_clean}"
        station_dir.mkdir(parents=True, exist_ok=True)
        # Simpan output per stasiun
        base_filename = f"FKLIM_{start_date_meta}_{end_date_meta}_{wmo_id}"
        df.to_csv(station_dir / f"{base_filename}_DAILY.csv", index=False)
        indices_reset.to_csv(station_dir / f"{base_filename}_INDICES.csv", index=False)
        if not trends.empty:
            trends.to_csv(station_dir / f"{base_filename}_TRENDS.csv", index=True)
        # Simpan untuk agregasi nasional
        DataBucket.append(df)
        if not trends.empty:
            SlopeBucket.append(trends.reset_index(drop=True))
        IndekBucket.append(indices_reset)  # ✅ FIX KRITIS: "IndekBucket" (bukan "IndicesBucket")
        print(f"  ✓ Berhasil: {len(indices_reset)} tahun data diproses, {len(trend_cols)} indeks untuk tren")
    except Exception as e:
        print(f"  ✗ Gagal total memproses stasiun {wmo_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

Memproses 110 stasiun dari direktori QC...

[1/110] Memproses stasiun 96001 (1981-01-01 - 2025-12-31)...
  ℹ️  Baseline curah hujan: 1991-2020 | QC: BASELINE_1991_2020
  ✓ Tren berhasil dihitung: 56 variabel
    Contoh nilai: NAME = Stasiun Meteorologi Maimun Saleh (non-numerik)
  ✓ Berhasil: 45 tahun data diproses, 47 indeks untuk tren

[2/110] Memproses stasiun 96009 (1981-01-01 - 2025-12-31)...
  ℹ️  Baseline curah hujan: 1991-2020 | QC: BASELINE_1991_2020
  ✓ Tren berhasil dihitung: 56 variabel
    Contoh nilai: NAME = Stasiun Meteorologi Malikussaleh (non-numerik)
  ✓ Berhasil: 44 tahun data diproses, 47 indeks untuk tren

[3/110] Memproses stasiun 96011 (1981-01-01 - 2025-12-31)...
  ℹ️  Baseline curah hujan: 1991-2020 | QC: BASELINE_1991_2020
  ✓ Tren berhasil dihitung: 56 variabel
    Contoh nilai: NAME = Stasiun Meteorologi Sultan Iskandar Muda (non-numerik)
  ✓ Berhasil: 44 tahun data diproses, 47 indeks untuk tren

[4/110] Memproses stasiun 96015 (1981-01-01 - 2025-12-31)...

In [17]:
# ==============================================================================
# AGREGASI NASIONAL (HANYA JIKA ADA DATA YANG BERHASIL DIPROSES)
# ==============================================================================
if not IndekBucket:
    print("\n❌ Tidak ada stasiun yang berhasil diproses. Pipeline dihentikan.")
    sys.exit(1)
    
print("\n" + "="*70)
print("MENGAGREGASI HASIL NASIONAL...")
print("="*70)

# Gabungkan semua data
try:
    dataRaw        = pd.concat(DataBucket, ignore_index=False) if DataBucket else pd.DataFrame()
    indicesdf      = pd.concat(IndekBucket, ignore_index=False) if IndekBucket else pd.DataFrame()
    meta_cols      = ['NAME', 'WMO_ID', 'START_DATA', 'END_DATA', 'YEAR', 'LAT', 'LON', 'ELE', 'PROV', 'REGENCY','baseline_period','qc_flag']
    data_cols      = [col for col in indicesdf.columns if col not in meta_cols]
    indicesdf_long = indicesdf.melt(id_vars=meta_cols, value_vars=data_cols, var_name='INDICES', value_name='VALUE')
    indicesdf_long = indicesdf_long.sort_values(by=['WMO_ID', 'YEAR', 'INDICES']).reset_index(drop=True)


    slopedf                 = pd.concat(SlopeBucket, ignore_index=False) if SlopeBucket else pd.DataFrame()
    slopedf                 = slopedf.reset_index().rename(columns={'index': 'STATS'})
    slopedf['STATS']        = slopedf['STATS'].replace({0: 'slope', 1: 'intercept', 2: 'r_value', 3: 'p_value', 4: 'std_err', 5: 'n_valid'})
    trendline               = slopedf[slopedf['STATS'] == 'slope']  # Tambah kolom VARIABLE untuk identifikasi
    # Trenline / 30 untuk perubahan per dekade
    trendline                 = trendline.copy()
    trendline_cols            = [col for col in trendline.columns if col not in ['NAME', 'WMO_ID', 'START_DATA', 'END_DATA', 'LAT', 'LON', 'ELE', 'PROV', 'REGENCY', 'STATS']]
    slopedf_long              = slopedf.copy()
    metacols                  = ['NAME', 'WMO_ID', 'START_DATA', 'END_DATA', 'LAT', 'LON', 'ELE', 'PROV', 'REGENCY', 'STATS']
    data_cols                 = [col for col in slopedf.columns if col not in metacols]
    slopedf_long              = slopedf_long.melt(id_vars=metacols, value_vars=data_cols, var_name='INDICES', value_name='VALUE')

    
    # Ambil tahun dari data yang berhasil diproses (lebih robust)
    if not indicesdf.empty and 'START_DATA' in indicesdf.columns and 'END_DATA' in indicesdf.columns:
        # Ambil dari baris pertama yang memiliki data valid
        valid_start = indicesdf['START_DATA'].dropna().iloc[0] if not indicesdf['START_DATA'].dropna().empty else "1981-01-01"
        valid_end = indicesdf['END_DATA'].dropna().iloc[0] if not indicesdf['END_DATA'].dropna().empty else "2024-12-31"
        start_date_meta = str(valid_start).split(' ')[0] if pd.notna(valid_start) else "1981-01-01"
        end_date_meta = str(valid_end).split(' ')[0] if pd.notna(valid_end) else "2024-12-31"
    else:
        # Fallback ke nilai default
        start_date_meta = "1981-01-01"
        end_date_meta = "2025-12-31"
    
    str_year = start_date_meta.split('-')[0]
    end_year = end_date_meta.split('-')[0]
    yearDir  = Path(indiceDir) / f"{str_year}_{end_year}"
    yearDir.mkdir(parents=True, exist_ok=True)
    
    # Simpan agregat nasional
    base_agg_name = f"FKLIM_AGGREGATE_{start_date_meta}_{end_date_meta}"
    
    if not dataRaw.empty:
        dataRaw.to_csv(yearDir / f"{base_agg_name}_DAILY.csv", index=False)
        print(f"✓ Data harian agregat disimpan: {len(dataRaw)} records")
    
    if not indicesdf.empty:
        indicesdf.to_csv(yearDir / f"{base_agg_name}_INDICES.csv", index=False)
        indicesdf_long.to_csv(yearDir / f"{base_agg_name}_INDICES_LONG.csv", index=False)
        with pd.ExcelWriter(yearDir / f"{base_agg_name}_INDICES.xlsx") as writer:
            indicesdf.to_excel(writer, sheet_name='INDICES', index=False)
            indicesdf_long.to_excel(writer, sheet_name='INDICES_LONG', index=False)
        print(f"✓ Indeks agregat disimpan: {len(indicesdf)} records, {len(indicesdf.columns)} kolom")
    
    if not slopedf.empty:
        slopedf_long.to_csv(yearDir / f"{base_agg_name}_TRENDS_LONG.csv", index=False)
        trendline.to_csv(yearDir / f"{base_agg_name}_TRENDS.csv", index=False)
        with pd.ExcelWriter(yearDir / f"{base_agg_name}_TRENDS.xlsx") as writer:
            trendline.to_excel(writer, sheet_name='SLOPE', index=False)
            slopedf_long.to_excel(writer, sheet_name='ALL_STATS', index=False)
        
        # Statistik tren untuk verifikasi
        slope_cols      = [col for col in slopedf.columns if col not in ['NAME', 'WMO_ID', 'START_DATA', 'END_DATA', 'LAT', 'LON', 'ELE', 'PROV', 'REGENCY']]
        valid_slopes    = slopedf[slope_cols].dropna(how='all')
        n_stations      = len(valid_slopes)
        n_variables     = len(slope_cols)
        n_valid_entries = valid_slopes.notna().sum().sum()
        
        print(f"✓ Tren agregat disimpan: {n_stations} stasiun, {n_variables} variabel")
        print(f"  Detail: {n_valid_entries} nilai slope valid dari total {n_stations * n_variables} kemungkinan")
    # 
    # Ringkasan QC
    if 'qc_flag' in indicesdf.columns:
        qc_summary = indicesdf['qc_flag'].value_counts().sort_values(ascending=False).to_dict()
        print("\n📊 Ringkasan QC Indeks Curah Hujan:")
        for flag, count in qc_summary.items():
            print(f"   • {flag}: {count} stasiun")
    
    print("\n✅ Pipeline selesai dengan sukses!")
    print(f"   Output tersedia di: {yearDir.absolute()}")
    
except Exception as e:
    print(f"\n❌ Gagal mengagregasi hasil nasional: {str(e)}")
    import traceback
    traceback.print_exc()
    sys.exit(1)


MENGAGREGASI HASIL NASIONAL...


/tmp/ipykernel_2695013/2190886619.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  indicesdf      = pd.concat(IndekBucket, ignore_index=False) if IndekBucket else pd.DataFrame()


✓ Data harian agregat disimpan: 1692483 records
✓ Indeks agregat disimpan: 4815 records, 61 kolom
✓ Tren agregat disimpan: 660 stasiun, 48 variabel
  Detail: 27756 nilai slope valid dari total 31680 kemungkinan

📊 Ringkasan QC Indeks Curah Hujan:
   • BASELINE_1991_2020: 3925 stasiun
   • NO_RAINFALL_DATA: 890 stasiun

✅ Pipeline selesai dengan sukses!
   Output tersedia di: /mnt/dataset/02_REPO_GITHUB_FIRMAN/Developing_Climate_Change_Indices_Station_Dataset/notebook/../02.output/01.indices/1981_2025
